# SO-101 cube pick-and-place · ACT on Colab
Select an A100 GPU runtime. Enable notebook access for the Colab secrets `github_token` and `hf_token`. Run the setup and smoke cells before starting the full run. The initial policy covers one task, red cube into blue box. Checkpoints save to Google Drive at the end of each training cell.

In [ ]:
import subprocess
import sys

if sys.version_info < (3, 12):
    raise RuntimeError("Select a Colab runtime with Python 3.12 or newer.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "lerobot[dataset,training]==0.6.1"],
    check=True,
)

In [ ]:
from pathlib import Path

import requests
import torch
from google.colab import drive, userdata
from huggingface_hub import snapshot_download
from lerobot.datasets import LeRobotDataset

PROJECT = "Jay-Aditya-16/so101-grasp-recovery"
DATASET = "caspar/so101_pickplace_sim_150"
REVISION = "b4fb342927bb0ee7dcbe4b90472a0c257ce4f5de"
TASK = "Pick up the red cube and put it in the blue box."
SEED = 1000

if not torch.cuda.is_available() or "A100" not in torch.cuda.get_device_name(0):
    raise RuntimeError("An A100 CUDA runtime is required for this recipe.")

response = requests.get(
    f"https://api.github.com/repos/{PROJECT}",
    headers={"Authorization": f"Bearer {userdata.get('github_token')}"},
    timeout=20,
)
response.raise_for_status()
if not response.json()["private"]:
    raise RuntimeError("Expected the project repository to be private.")
del response

root = Path(snapshot_download(
    DATASET, repo_type="dataset", revision=REVISION,
    token=userdata.get("hf_token"), local_dir="/content/so101-dataset",
))
dataset = LeRobotDataset(DATASET, root=root, video_backend="pyav")
if (dataset.num_episodes, len(dataset)) != (117, 20124):
    raise RuntimeError("Unexpected dataset size.")
for index in (0, len(dataset) // 2, len(dataset) - 1):
    sample = dataset[index]
    if (
        sample["task"] != TASK
        or tuple(sample["observation.state"].shape) != (6,)
        or tuple(sample["action"].shape) != (6,)
        or any(
            tuple(sample[f"observation.images.{camera}"].shape) != (3, 240, 320)
            for camera in ("top", "wrist")
        )
    ):
        raise RuntimeError(f"Dataset contract mismatch at frame {index}.")
print(f"{len(dataset):,} frames across {dataset.num_episodes} episodes validated by metadata and sample reads.")
drive.mount("/content/drive")

In [ ]:
import shutil
from datetime import datetime, timezone

TRAINER = shutil.which("lerobot-train")
if not TRAINER:
    raise RuntimeError("lerobot-train is missing. Run the installation cell.")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/so101-act")
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

def train_act(label, steps, batch_size, save_freq):
    run = f"{label}-{datetime.now(timezone.utc):%Y%m%dT%H%M%S}"
    output = Path("/content/so101-training") / run
    command = [
        TRAINER,
        f"--dataset.repo_id={DATASET}",
        f"--dataset.root={root}",
        f"--dataset.revision={REVISION}",
        "--dataset.video_backend=pyav",
        "--policy.type=act",
        "--policy.device=cuda",
        "--policy.push_to_hub=false",
        "--policy.chunk_size=50",
        "--policy.n_action_steps=20",
        f"--batch_size={batch_size}",
        "--num_workers=2",
        f"--steps={steps}",
        f"--save_freq={save_freq}",
        "--env_eval_freq=0",
        "--wandb.enable=false",
        f"--seed={SEED}",
        f"--output_dir={output}",
    ]
    subprocess.run(command, check=True)
    policy = output / "checkpoints" / "last" / "pretrained_model"
    if not (policy / "model.safetensors").is_file():
        raise RuntimeError(f"Missing checkpoint at {policy}")
    archive = shutil.make_archive(str(output.parent / run), "zip", policy.parent, policy.name)
    saved = shutil.copy2(archive, DRIVE_OUTPUT)
    print(f"Saved inference checkpoint and normalization processors to {saved}")
    return saved

smoke_checkpoint = train_act("smoke", steps=20, batch_size=4, save_freq=20)

## Train the baseline
Run this cell after the smoke checkpoint saves. The 20,000-step run starts from a fresh ACT policy and writes intermediate checkpoints to Colab's local disk. The final inference checkpoint is copied to Drive. If the runtime is interrupted, intermediate files are available only while that runtime persists.

In [ ]:
baseline_checkpoint = train_act("single-cube-act", steps=20_000, batch_size=16, save_freq=5_000)